# BAMoE — Bias-Aware Mixture of Experts for Time Series Forecasting

Run all five experiments end-to-end:
1. **Preliminary** — single-bias Transformer comparison
2. **Main results** — BAMoE vs. baselines
3. **Ablation** — expert diversity, routing mechanism, K-sweep
4. **Interpretability** — routing dynamics and attention patterns
5. **Efficiency** — parameter count vs. performance

> **Before running:** make sure the repo is public (or you have a token), and that `Runtime → Change runtime type` is set to **GPU**.

## 0 · Configuration
Edit the values in this cell before running anything else.

In [ ]:
# ── Repository ────────────────────────────────────────────────────────────────
GITHUB_REPO   = "https://github.com/hingma/BAMoE.git"
BRANCH        = "main"

# ── Paths ─────────────────────────────────────────────────────────────────────
# Set USE_DRIVE=True to persist checkpoints/results across Colab sessions.
USE_DRIVE     = False
DRIVE_DIR     = "/content/drive/MyDrive/BAMoE"   # ignored when USE_DRIVE=False

# ── Default model hyperparameters ─────────────────────────────────────────────
CFG = dict(
    seq_len     = 336,
    d_model     = 128,
    n_heads     = 8,
    n_layers    = 3,
    d_ff        = 256,
    dropout     = 0.1,
    patch_len   = 16,
    stride      = 8,
    # BAMoE-specific
    expert_types        = "causal,local,periodic,global",
    top_k               = 2,
    routing             = "learned_sparse",
    load_balance_coef   = 0.01,
    local_window        = 3,
    periodic_period     = 12,
    # Training
    batch_size    = 128,
    learning_rate = 1e-4,
    train_epochs  = 20,
    patience      = 5,
    weight_decay  = 1e-4,
    lradj         = "cosine",
    # Data
    features = "M",
    target   = "OT",
    num_workers = 2,
)

## 1 · Environment setup

In [ ]:
import subprocess, sys, os

IN_COLAB = "google.colab" in sys.modules

# GPU info
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                    "--format=csv,noheader"], capture_output=True, text=True)
print("GPU :", r.stdout.strip() if r.returncode == 0 else "none — running on CPU")
print("Python:", sys.version.split()[0])

# Optional: mount Google Drive
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print("Drive mounted →", DRIVE_DIR)

## 2 · Install dependencies

In [ ]:
!pip install torch numpy pandas scikit-learn matplotlib seaborn scipy tqdm --quiet

## 3 · Clone repository

In [ ]:
WORK_DIR = DRIVE_DIR if (IN_COLAB and USE_DRIVE) else "/content/BAMoE"

if os.path.isdir(os.path.join(WORK_DIR, ".git")):
    print("Repo already cloned — pulling latest changes...")
    !git -C {WORK_DIR} pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {GITHUB_REPO} {WORK_DIR}

%cd {WORK_DIR}
print("Working directory:", os.getcwd())

## 4 · Download datasets

In [ ]:
!bash scripts/download_data.sh ./data

## 5 · Helper: build args
Converts a plain dict into the `argparse.Namespace` that every experiment expects.

In [ ]:
import argparse, random, numpy as np, torch

def make_args(overrides: dict) -> argparse.Namespace:
    """Merge CFG defaults with per-experiment overrides."""
    d = dict(
        # paths
        root_path   = "./data",
        data_path   = "",
        checkpoints = "./checkpoints",
        results     = "./results",
        exp_name    = "",
        resume      = True,
        use_gpu     = 1,
        gpu         = 0,
        # model identity
        model     = "BAMoE",
        bias_type = "global",
        **CFG,
    )
    d.update(overrides)

    # Auto-generate exp_name if not provided
    if not d["exp_name"]:
        if d["model"] == "SingleBias":
            tag = f"SingleBias_{d['bias_type']}"
        else:
            k = len(d["expert_types"].split(","))
            tag = f"BAMoE_K{k}_{d['routing']}"
        d["exp_name"] = f"{tag}_{d['data']}_sl{d['seq_len']}_pl{d['pred_len']}"

    return argparse.Namespace(**d)


def set_seed(seed=2024):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def run_one(overrides: dict):
    """Train + test a single configuration. Returns (mse, mae, crps, mase)."""
    from exp.exp_forecast import ExpForecast
    set_seed()
    args = make_args(overrides)
    print(f"\n{'='*60}\n{args.exp_name}")
    exp = ExpForecast(args)
    exp.train()
    return exp.test()


print("Helper functions ready.")

---
## Experiment 1 — Preliminary: Temporal Inductive Bias Analysis
Train four single-bias Transformers across datasets and horizons.
Adjust `EXP1_DATASETS` and `EXP1_PRED_LENS` to run a subset.

In [ ]:
EXP1_BIAS_TYPES = ["global", "causal", "local", "periodic"]
EXP1_DATASETS   = ["ETTh1", "ETTh2", "Weather", "Traffic"]   # add ETTm1/m2, Electricity, Exchange
EXP1_PRED_LENS  = [96, 192, 336, 720]

for bias in EXP1_BIAS_TYPES:
    for data in EXP1_DATASETS:
        for pl in EXP1_PRED_LENS:
            run_one(dict(model="SingleBias", bias_type=bias, data=data, pred_len=pl))

---
## Experiment 2 — Main BAMoE Results

In [ ]:
EXP2_DATASETS  = ["ETTh1", "ETTh2", "ETTm1", "ETTm2", "Weather", "Traffic", "Electricity", "Exchange"]
EXP2_PRED_LENS = [96, 192, 336, 720]

for data in EXP2_DATASETS:
    for pl in EXP2_PRED_LENS:
        run_one(dict(model="BAMoE", data=data, pred_len=pl))

---
## Experiment 3 — Ablation Studies
### 3a · Expert diversity

In [ ]:
EXP3_DATASETS  = ["ETTh1", "Weather", "Traffic"]
EXP3_PRED_LENS = [96, 192, 336, 720]

DIVERSITY_VARIANTS = {
    "homo_causal"   : "causal,causal,causal,causal",
    "homo_local"    : "local,local,local,local",
    "homo_periodic" : "periodic,periodic,periodic,periodic",
    "homo_global"   : "global,global,global,global",
    "K1"            : "causal",
    "hetero"        : "causal,local,periodic,global",
}

for tag, etypes in DIVERSITY_VARIANTS.items():
    for data in EXP3_DATASETS:
        for pl in EXP3_PRED_LENS:
            run_one(dict(model="BAMoE", expert_types=etypes, data=data, pred_len=pl,
                         exp_name=f"ablation_diversity_{tag}_{data}_pl{pl}"))

### 3b · Routing mechanism

In [ ]:
ROUTING_VARIANTS = ["uniform", "random", "top1", "learned_sparse", "dense"]

for routing in ROUTING_VARIANTS:
    k = 1 if routing == "top1" else 2
    for data in EXP3_DATASETS:
        for pl in EXP3_PRED_LENS:
            run_one(dict(model="BAMoE", routing=routing, top_k=k,
                         data=data, pred_len=pl,
                         exp_name=f"ablation_routing_{routing}_{data}_pl{pl}"))

### 3c · Number of experts K

In [ ]:
K_VARIANTS = {
    "K2" : "causal,global",
    "K3" : "causal,local,periodic",
    "K4" : "causal,local,periodic,global",
    "K6" : "causal,local,periodic,global,causal,local",
    "K8" : "causal,local,periodic,global,causal,local,periodic,global",
}

for tag, etypes in K_VARIANTS.items():
    for data in EXP3_DATASETS:
        for pl in EXP3_PRED_LENS:
            run_one(dict(model="BAMoE", expert_types=etypes,
                         data=data, pred_len=pl,
                         exp_name=f"ablation_Ksweep_{tag}_{data}_pl{pl}"))

---
## Experiment 4 — Interpretability Analysis
Requires completed Experiment 2 checkpoints.

In [ ]:
from exp.exp_interpretability import ExpInterpretability

EXP4_DATASETS  = ["ETTh1", "ETTh2", "Weather", "Traffic"]
EXP4_PRED_LENS = [96, 192, 336, 720]

for data in EXP4_DATASETS:
    for pl in EXP4_PRED_LENS:
        args = make_args(dict(model="BAMoE", data=data, pred_len=pl))
        ExpInterpretability(args).run()

---
## Results — Summary Table

In [ ]:
import pandas as pd

df = pd.read_csv("results/summary.csv")

# Average MSE across horizons per (model, data) for a quick overview
pivot = (
    df.groupby(["model", "data", "pred_len"])[["mse", "mae", "crps", "mase"]]
    .mean()
    .round(4)
)
pd.set_option("display.max_rows", 200)
pivot

In [ ]:
import matplotlib.pyplot as plt

# Bar chart: average MSE per model across all datasets and horizons
avg = df.groupby("model")["mse"].mean().sort_values()
fig, ax = plt.subplots(figsize=(max(6, len(avg) * 0.8), 4))
avg.plot.bar(ax=ax, color="steelblue", edgecolor="white")
ax.set_ylabel("Average MSE")
ax.set_title("Average MSE per model (all datasets × horizons)")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig("results/avg_mse_bar.pdf", dpi=150)
plt.show()

---
## Download Results

In [ ]:
import shutil, os

# Bundle results (CSV + figures) and checkpoints into zip archives
shutil.make_archive("bamoe_results",     "zip", "results")
shutil.make_archive("bamoe_checkpoints", "zip", "checkpoints")

print(f"results     : {os.path.getsize('bamoe_results.zip')     / 1e6:.1f} MB")
print(f"checkpoints : {os.path.getsize('bamoe_checkpoints.zip') / 1e6:.1f} MB")

if IN_COLAB:
    from google.colab import files
    files.download("bamoe_results.zip")
    files.download("bamoe_checkpoints.zip")
else:
    print("Not on Colab — zip files saved in the working directory.")